# 03 — Earth observation GeoAI: Major TOM + NASA GIBS

**Atlas source reading copy · local-server**

[Course guide](index.ipynb) · [Pinned original](https://github.com/jltobias/JupyterLite-GeoLibre-GeoAI/blob/777fa8965068b1e26b982162f17254ee2df7ec84/book/notebooks/03_earth_observation_geoai.ipynb)



Remote services, tiles, CDN assets or package retrieval are subject to network/CORS availability. Use the source local/server environment for native GIS, shell, installed project or legacy dependencies; Lab provides a reading/editing copy. Widget/anywidget/LexCube branches need their source extensions; the Atlas shell supplies only Pyodide.

**Provenance:** `book/notebooks/03_earth_observation_geoai.ipynb` at `777fa8965068b1e26b982162f17254ee2df7ec84`; original SHA-256 `145e540181e14be3a7569a03f448306562a14352399fbe29d99d2556d453f076`. Original author material incorporated at repository owner request; no general license inferred. Third-party terms remain separate.

**Verification:** imported by static inspection; saved upstream outputs cleared. Runtime compatibility is separate from executed verification. See the course guide for dependencies, excluded assets, aliases and original credits.


# 03 — Earth observation GeoAI: Major TOM + NASA GIBS

**[🚀 Launch this notebook live](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/lite/lab/index.html?path=03_earth_observation_geoai.ipynb)**

This lab keeps the AI-ready **Major TOM Core** Sentinel-2, Copernicus DEM, and ESA WorldCover layers in GeoLibre, then runs a browser-safe unsupervised classification on a fixed **NASA GIBS MODIS Terra true-color** image.

> **Why the split?** Major TOM's Sentinel-2 chip is ZSTD-compressed. GeoLibre's browser raster engine can stream the COG, but the Rasterio/GDAL build currently available in Pyodide does not include the GDAL ZSTD GeoTIFF codec. Reading that TIFF with `rasterio.read()` raises `CPLE_AppDefinedError: ZSTD compression support is not configured`. The ML section therefore uses a JPEG response from NASA GIBS, avoiding GDAL compression codecs entirely.

> **JupyterLite compatibility:** maps use `geolibre_lite.LiteMap`, which avoids the localhost socket required by upstream `geolibre.Map`.


## Start here: the question behind the code

**Can a browser learn patterns from a small satellite image?**

**What you will learn:** fetch → reshape → learn → map. The numbered reading guides below follow the actual code cells. Read a guide, predict the output, then run its cell with **Shift+Enter**. Run from top to bottom; changing an upstream value requires rerunning the cells that use it.

**GeoAI connection:** Browser machine learning: MiniBatch K-Means groups RGB appearance; no labeled land-cover model is trained.

**Why this is useful:** You can follow an entire image-to-feature-to-model-to-map pipeline without hiding the relationship between pixels and predictions.

**Where the work happens:** A small NASA JPEG is decoded and clustered in Pyodide. The separate Major TOM COG layers render in GeoLibre.

![Pixels become model features. A fixed date and bounding box anchor a compact image sample. Each pixel becomes a row with red, green and blue feature values. Six clusters describe color similarity, with arbitrary IDs. Restore geographic position, then compare original image and grouped regions.](assets/learning/03/infographic.svg)

### Watch the idea unfold

[[Source image continuation: Preview: Pixels become model features — 32-second captioned walkthrough](https://github.com/jltobias/JupyterLite-GeoLibre-GeoAI/blob/777fa8965068b1e26b982162f17254ee2df7ec84/book/notebooks/assets/learning/03/poster.png)](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/lite/files/assets/learning/03/watch.html)

[▶ Watch the captioned mini-video and read its transcript](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/lite/files/assets/learning/03/watch.html) · [Launch this notebook](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/lite/lab/index.html?path=03_earth_observation_geoai.ipynb) · [Read this Book chapter](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/notebooks/03_earth_observation_geoai.html) · [Python and browser learning guide](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/learning-guide.html)

The video is a silent four-step teaching storyboard with player controls and captions. Open the companion page to play, pause, seek or download it. The local copy is bundled under `assets/learning/03/`; downloaded media can be played offline. The infographic and the text below cover the same ideas without a video player.

**Video transcript:**

1. **Fetch (0–8 s):** A fixed date and bounding box anchor a compact image sample. Code landmark: `source_image.resize((128, 128))`.

2. **Reshape (8–16 s):** Each pixel becomes a row with red, green and blue feature values. Code landmark: `rgb.reshape(-1, 3)`.

3. **Learn (16–24 s):** Six clusters describe color similarity, with arbitrary IDs. Code landmark: `model.fit_predict(pixels)`.

4. **Map (24–32 s):** Restore geographic position, then compare original image and grouped regions. Code landmark: `from_bounds(*BBOX, w, h)`.


## See the connections

![Earth observation: pixels to mapped patterns: Imagery: Reflectance appearance varies with clouds; Learning: RGB similarity groups colors, not land uses; Geometry: A known bbox places image cells on Earth; Check: Validate classes against independent labels](assets/03_mindmap.svg)

### Code guide 1 — Prepare the browser toolbox

**Read the Python:** `import` makes a package available to Python. `sys.platform` identifies the runtime; the indented `if` block runs only in Pyodide. `await` waits for package loading. `LiteMap as Map` gives the browser-compatible map class a shorter name. Installing a package and importing its functions are separate steps.

**Look for:** Imports complete before you run the next cell. The first run can take longer while packages download.

**Pause and try:** Find the alias `Map`. If a later cell says a name is undefined, rerun this setup cell first.


In [ ]:
import sys
if sys.platform == "emscripten":
    import micropip
    await micropip.install(["geolibre==3.0.0", "pyodide-http"])

from geolibre_lite import LiteMap as Map
assert Map.__module__ == "geolibre_lite", Map.__module__
print("Map implementation:", Map.__module__)


## Stream AI-ready Major TOM layers in GeoLibre

These COGs stay remote and are rendered by GeoLibre in the browser. We deliberately do **not** decode the ZSTD Sentinel-2 TIFF with Pyodide Rasterio.


### Code guide 2 — Stream three complementary Earth layers

**Read the Python:** The f-strings reuse a base address to form three COG URLs. `bands=[4, 3, 2]` chooses the displayed Sentinel-2 channels; `rescale` controls their appearance. Elevation and land cover provide different context for the same example chip.

**Look for:** A GeoLibre project with imagery, terrain and land-cover layers; these layers are separate from the following NASA image experiment.

**Pause and try:** Toggle each layer and state what its values represent before comparing their colors.


In [ ]:
BASE = "https://data.source.coop/major-tom/core/DATA/42"
S2 = f"{BASE}/s2/data.tif"
DEM = f"{BASE}/dem/data.tif"
WC = f"{BASE}/wc/data.tif"

major_tom = Map(center=(8.0, 48.0), zoom=5, height="620px")
major_tom.add_cog(S2, name="Major TOM Sentinel-2 RGB", bands=[4, 3, 2], rescale=[0, 3000])
major_tom.add_cog(DEM, name="Major TOM Copernicus DEM", colormap="terrain")
major_tom.add_cog(WC, name="Major TOM ESA WorldCover")
major_tom


## Browser-safe EO classification from NASA GIBS

NASA GIBS exposes imagery through standards-based WMS/WMTS services. Here we request a fixed MODIS Terra true-color image over the San Francisco Bay region as JPEG, resize it to 128×128 pixels, and cluster RGB values with MiniBatch K-Means. The WMS bounding box gives us the georeferencing needed to map the cluster polygons.


### Code guide 3 — Request a small georeferenced image

**Read the Python:** `BBOX` records west, south, east, north. The WMS request fixes an image date and size. `response.content` is a sequence of bytes; `BytesIO` lets Pillow open it like a file, then `.convert('RGB')` creates three color channels.

**Look for:** A 512 by 512 NASA GIBS image, with the selected date printed beside its size.

**Pause and try:** Keep the same bounding box if you compare dates. Why would a changed extent confuse a change analysis?


In [ ]:
import io
import requests
import numpy as np
from PIL import Image
from IPython.display import display

if sys.platform == "emscripten":
    import pyodide_http
    pyodide_http.patch_all()

GIBS_WMS = "https://gibs.earthdata.nasa.gov/wms/epsg4326/best/wms.cgi"
BBOX = (-123.0, 37.0, -121.0, 39.0)  # west, south, east, north
DATE = "2024-07-15"
params = {
    "service": "WMS",
    "request": "GetMap",
    "version": "1.1.1",
    "layers": "MODIS_Terra_CorrectedReflectance_TrueColor",
    "styles": "",
    "format": "image/jpeg",
    "transparent": "false",
    "srs": "EPSG:4326",
    "bbox": ",".join(map(str, BBOX)),
    "width": 512,
    "height": 512,
    "time": DATE,
}

response = requests.get(GIBS_WMS, params=params, timeout=120)
response.raise_for_status()
source_image = Image.open(io.BytesIO(response.content)).convert("RGB")
print("GIBS image:", source_image.size, "date:", DATE)
display(source_image)


### Code guide 4 — Convert pixels into a learning table

**Read the Python:** Resizing makes the problem small enough for a browser. Dividing by 255 puts colors on a 0–1 scale. `reshape(-1, 3)` makes one row per pixel and three feature columns. MiniBatch K-Means assigns six color groups; the last reshape restores image rows and columns.

**Look for:** A 128 by 128 label grid and counts for each cluster ID.

**Pause and try:** Explain why the model can group similar colors without knowing the words forest, water or cloud.


In [ ]:
from sklearn.cluster import MiniBatchKMeans

analysis_image = source_image.resize((128, 128), Image.Resampling.BILINEAR)
rgb = np.asarray(analysis_image, dtype=np.float32) / 255.0
pixels = rgb.reshape(-1, 3)

model = MiniBatchKMeans(
    n_clusters=6,
    random_state=42,
    n_init=10,
    batch_size=2048,
)
labels = model.fit_predict(pixels).reshape(rgb.shape[:2]).astype("int16")
np.unique(labels, return_counts=True)


### Code guide 5 — Put the model labels back on Earth

**Read the Python:** `from_bounds` connects array rows and columns to the original bounding box. `shapes` traces same-label regions into polygons. `dissolve` groups polygons with the same cluster ID; `simplify(0.01)` reduces boundary detail in degrees.

**Look for:** One row per cluster with spatial geometry; simplified boundaries may lose small details.

**Pause and try:** Why should a detailed measurement use the original pixels or validated geometry instead of this simplified display?


In [ ]:
from rasterio.features import shapes
from rasterio.transform import from_bounds
import geopandas as gpd
from shapely.geometry import shape

h, w = labels.shape
transform = from_bounds(*BBOX, w, h)
records = [
    {"cluster": int(value), "geometry": shape(geom)}
    for geom, value in shapes(labels, transform=transform)
]

classes = gpd.GeoDataFrame(records, crs="EPSG:4326")
classes = classes.dissolve(by="cluster").reset_index()
classes["geometry"] = classes.geometry.simplify(0.01, preserve_topology=True)
classes[["cluster", "geometry"]]


### Code guide 6 — Display the geographic prediction

**Read the Python:** The cluster column selects the polygon color and `fillOpacity` keeps the basemap visible. The f-string places the acquisition date into the layer name so the map carries temporal context.

**Look for:** A map of RGB appearance groups over the requested region.

**Pause and try:** Find a bright image region and inspect its cluster; name two possible causes for that brightness.


In [ ]:
ml_map = Map(center=(-122.0, 38.0), zoom=7, height="650px")
ml_map.add_choropleth(
    classes,
    column="cluster",
    name=f"MODIS RGB K-Means — {DATE}",
    class_count=6,
    colormap="turbo",
    scheme="equal-interval",
    fillOpacity=0.55,
)
ml_map


## Interpretation

These are **RGB appearance clusters**, not semantic land-cover classes. Their purpose is to demonstrate a reproducible browser-native GeoAI pattern: retrieve EO imagery, convert pixels to features, fit an unsupervised model, georeference model output, and inspect the result in GeoLibre. A production workflow would use calibrated spectral bands, cloud masks, validation labels, and the full GeoAI/PyTorch stack where appropriate.

## Data & software citations

- Major TOM Core: https://source.coop/major-tom/core — AI-ready multimodal EO chips; example tile 42 supplies Sentinel-2, Copernicus DEM, and ESA WorldCover layers.
- Major TOM citation: Francis, A. & Czerkawski, M. (2024), *Major TOM: Expandable Datasets for Earth Observation*, IGARSS 2024, https://doi.org/10.1109/IGARSS53475.2024.10640760.
- NASA Global Imagery Browse Services (GIBS): https://nasa-gibs.github.io/gibs-api-docs/ — layer `MODIS_Terra_CorrectedReflectance_TrueColor`, fixed example date 2024-07-15.
- MODIS/Terra imagery is provided through NASA GIBS, part of NASA's Earth Science Data and Information System (ESDIS).
- Copernicus Sentinel-2 / Copernicus DEM and ESA WorldCover attribution follows the Major TOM source collection.
- GeoAI: Wu, Q. (2026), JOSS 11(118), 9605. https://doi.org/10.21105/joss.09605
- GeoLibre: https://geolibre.app/


### Code guide 7 — Load the plotting tools

**Read the Python:** `from IPython.display import display` lets a cell show objects explicitly. In Pyodide, `await loadPackage('matplotlib')` loads a compiled plotting package. The `if` block is skipped in ordinary Python, where the package must already be installed.

**Look for:** This preparation cell may show no figure; the next cell creates it.

**Pause and try:** Keep this cell before the chart cell when you rerun the notebook after a kernel restart.


In [ ]:
from IPython.display import display
import sys
if sys.platform == "emscripten":
    from pyodide import loadPackage
    await loadPackage("matplotlib")

## Connect image appearance, labels and class size

Use one categorical palette for the label image and its bars. The image extent is the requested WMS bounding box; row zero is north. The bar chart measures pixel share, not geodesic area.

### Code guide 8 — Read image, label and count together

**Read the Python:** `extent` uses west/east/south/north for plotting; `origin='upper'` places the first row at the north. A categorical palette links the label image to its bars. `bincount` counts label IDs, then division turns counts into percentages.

**Look for:** Three linked panels: appearance, clusters, and pixel share. Pixel share is not an equal-area land-cover statistic.

**Pause and try:** Change the number of clusters only if you also update the palette, boundaries and count length consistently.


In [ ]:
from cognition import style_plots, COLORS
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
style_plots()
fig, axes = plt.subplots(1, 3, figsize=(13, 4), layout="constrained")
extent = [BBOX[0], BBOX[2], BBOX[1], BBOX[3]]
axes[0].imshow(rgb, extent=extent, origin="upper")
axes[0].set_title(f"MODIS appearance: {DATE}")
axes[1].imshow(labels, extent=extent, origin="upper", cmap=ListedColormap(COLORS),
               norm=BoundaryNorm(np.arange(7)-.5, 6), interpolation="nearest")
axes[1].set_title("Six RGB clusters (not land cover)")
for ax in axes[:2]:
    ax.set(xlabel="Longitude", ylabel="Latitude")
counts = np.bincount(labels.ravel(), minlength=6)
axes[2].bar(range(6), 100 * counts / counts.sum(), color=COLORS)
axes[2].set(xlabel="Cluster ID", ylabel="Pixels (%)", title="How much of the image is each color?")
plt.show()

**Test a hypothesis:** identify a visually bright cluster. Could it contain cloud, built surfaces, or both? A real validation set is needed before assigning a land-cover label. Continue in lab 13 to compare images with Astra.

## Make sense of the result

![Blue pixels must all be water! The model learned colors, not the meaning of water. I need independent labels to test that claim.](assets/learning/03/cartoon.svg)

**Cartoon transcript:** Blue pixels must all be water! → The model learned colors, not the meaning of water. → I need independent labels to test that claim.

**Explain it to a colleague:** Can cloud and built surfaces land in the same bright cluster?

<details><summary>Check your reasoning</summary>

Yes. RGB similarity can combine different surface types; semantic labels require additional evidence.

</details>

Write three sentences: **I used…** (source and units), **the code computed…** (method), and **I can conclude…** (plus a remaining question). Point to the map, chart or table that supports each claim.

All new teaching illustrations are original schematics. Their marks and fictional dialogue are explanatory, not observations, fitted predictions or captured AI conversations. The existing citation cells identify the analytical data.


## GeoLibre creator and software attribution

GeoLibre is developed by **Qiusheng Wu and contributors**. Recommended citation: Wu, Q. (2026). *GeoLibre: A lightweight, cloud-native GIS platform for visualizing, exploring, and analyzing geospatial data.* Zenodo. [doi:10.5281/zenodo.20785400](https://doi.org/10.5281/zenodo.20785400). See [the official citation guidance](https://geolibre.app/citation/) and [source and MIT license](https://github.com/opengeos/GeoLibre). These browser lessons pin the Python builders to `geolibre==3.0.0`; the independently hosted viewer may change. This repository supplies the JupyterLite adapter and teaching materials, not the upstream GIS. Keep the original dataset, demo-author and basemap credits when sharing a result.
